# GL Code Automation - UPS billing (CooperSurgical / Belimo)

1. Fill column M using the rules from the GL code logic doc (v1)
2. Compare with the completed files (answer key)
3. Look for patterns and write better rules (v2)

The final rules are also in `GL_Code_Apply.ipynb`, which is the one to run on new data.

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)


def col_idx(letter):
    # excel column letter -> 0-based index ('M' -> 12, 'AQ' -> 42)
    n = 0
    for ch in letter.strip().upper():
        n = n * 26 + (ord(ch) - 64)
    return n - 1


def read_ups_csv(path):
    # UPS files have no header row, read everything as text so nothing gets reformatted
    for enc in ["utf-8", "latin1"]:
        try:
            df = pd.read_csv(
                path, header=None, dtype=str, keep_default_na=False, encoding=enc
            )
            df.attrs["encoding"] = enc
            return df
        except UnicodeDecodeError:
            pass
    raise ValueError("could not read " + str(path))


def normalize_gl(code):
    # long dash -> '--', and 'Split 654...' / 'Split-654...' -> 'Split-654...'
    if not isinstance(code, str):
        return code
    code = code.strip()
    code = re.sub(r"[\u2014\u2015]", "--", code)
    code = re.sub(r"[\u2010\u2011\u2012\u2013]", "-", code)
    code = re.sub(r"^split[\s\-_:]*", "Split-", code, flags=re.I)
    return code


def get_col(df, c):
    return df[col_idx(c)].astype(str).str.strip()


def make_pattern(vals, word=False):
    if isinstance(vals, str):
        vals = [vals]
    alt = "|".join(re.escape(str(v)) for v in vals)
    if word:
        return r"(?<![A-Za-z0-9])(?:" + alt + r")(?![A-Za-z0-9])"
    return "(?:" + alt + ")"

In [2]:
INPUT_DIR = Path("input")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


def detect_company(path):
    name = Path(path).name.upper()
    if "CSI" in name or "COOPER" in name:
        return "CSI"
    if "BELIMO" in name:
        return "Belimo"
    raise ValueError("can't tell the company from file name: " + Path(path).name)


def clean_name(path):
    # UPS file names come with spaces or underscores ("Not Completed" / "Not_Completed")
    return re.sub(r"[\s_]+", "_", Path(path).name)


def is_answer_key(path):
    name = clean_name(path)
    return "_Completed_" in name and "Not_Completed" not in name


raw, truth = {}, {}
for p in sorted(INPUT_DIR.glob("*.csv")):
    comp = detect_company(p)
    if is_answer_key(p):
        truth[comp] = read_ups_csv(p)[col_idx("M")].map(normalize_gl).values
    else:
        raw[comp] = read_ups_csv(p)

for comp in raw:
    print(comp, raw[comp].shape, "| answer key:", comp in truth)

Belimo (8705, 246) | answer key: True
CSI (3263, 244) | answer key: True


In [3]:
# the completed file should be the same as the original except column M
for comp in raw:
    key = read_ups_csv(
        next(
            p
            for p in INPUT_DIR.glob("*.csv")
            if is_answer_key(p) and detect_company(p) == comp
        )
    )
    other = [c for c in raw[comp].columns if c != col_idx("M")]
    print(
        comp,
        "cells different outside column M:",
        int((key[other] != raw[comp][other]).to_numpy().sum()),
    )

Belimo cells different outside column M: 0
CSI cells different outside column M: 0


## v1: rules from the doc

Notes on how I read the doc:
- GL codes end with two short dashes `--` (the long dash in the PDF is the same thing)
- doc says AQ = TAAX, the data only has TAX
- RMA / Lit use whole-word matching, otherwise "SHARMA" and "Facility" match
- SO55000 rules: the city check uses the sender city (BS) instead of CA. The 727 rows that were already filled in the CSI file all match with BS but only 30 match with CA
- Belimo "BP does not contain Warren or BP does not contain Belimo" is read as BP contains neither
- Belimo rules ending with "and Column CD" have no value for CD, so they are turned off

In [4]:
# a rule is a list of conditions (col, op, value), all of them have to be true
# "P|Q" means column P or column Q
OPS = [
    "eq",
    "in",
    "contains",
    "word",
    "not_contains",
    "contains_any",
    "not_contains_any",
    "eq_col",
    "ne_col",
]


def cond_mask(df, cols, op, val):
    if op in ["eq_col", "ne_col"]:
        # compare two columns, e.g. sender city BS vs receiver city CA
        a = get_col(df, cols).str.lower()
        b = get_col(df, val).str.lower()
        return np.asarray(a == b) if op == "eq_col" else np.asarray(a != b)
    masks = []
    for c in cols.split("|"):
        s = get_col(df, c)
        if op == "eq":
            m = s.str.lower() == str(val).strip().lower()
        elif op == "in":
            m = s.str.lower().isin([str(v).strip().lower() for v in val])
        elif op == "word":
            m = s.str.contains(make_pattern(val, word=True), case=False, regex=True)
        else:  # contains / not_contains / contains_any / not_contains_any
            m = s.str.contains(make_pattern(val), case=False, regex=True)
        masks.append(np.asarray(m))
    if op.startswith("not_"):
        return ~np.logical_or.reduce(masks)
    return np.logical_or.reduce(masks)


def rule_mask(df, rule):
    m = np.ones(len(df), dtype=bool)
    for cols, op, val in rule["when"]:
        m &= cond_mask(df, cols, op, val)
    return m


def match_rules(df, rules):
    # for each row pick the matching rule with the highest priority
    # (ties -> the one listed first). conflict = top-priority rules disagree on the GL
    rules = [r for r in rules if r.get("enabled", True)]
    prio = np.array([r["priority"] for r in rules])
    gls = np.array([r["gl"] for r in rules], dtype=object)
    ids = np.array([r["id"] for r in rules], dtype=object)
    hits = np.column_stack([rule_mask(df, r) for r in rules])
    p = np.where(hits, prio, -1)
    best = p.max(axis=1)
    top = hits & (p == best[:, None])
    win = top.argmax(axis=1)
    matched = hits.any(axis=1)
    return pd.DataFrame(
        {
            "gl": np.where(matched, gls[win], ""),
            "rule": np.where(matched, ids[win], ""),
            "conflict": np.array([len(set(gls[row])) > 1 for row in top]),
            "all_rules": [";".join(ids[row]) for row in hits],
        }
    )

In [5]:
SO_CITY = "BS"  # change to "CA" to follow the doc literally

RULES_V1 = {}
RULES_V1["CSI"] = [
    {
        "id": "CSI-01",
        "when": [("AQ", "eq", "SVC")],
        "gl": "65415-5500--CSI_U100-L529--",
    },
    {
        "id": "CSI-02",
        "when": [("AQ", "eq", "PRT")],
        "gl": "65415-5500--CSI_U100-L529--",
    },
    {
        "id": "CSI-03",
        "when": [("AQ", "eq", "AUD")],
        "gl": "65415-5500--CSI_U100-L529--",
    },
    {
        "id": "CSI-04",
        "when": [("AQ", "eq", "TAX")],
        "gl": "65420-3000--CSI_U100-L527--",
    },
    {
        "id": "CSI-05",
        "when": [("P", "eq", "83350-15")],
        "gl": "65415-8401--CSI_M100-L510--",
    },
    {
        "id": "CSI-06",
        "when": [("P", "eq", "83350-60")],
        "gl": "65415-6000--CSI_U100-L529--",
    },
    {
        "id": "CSI-07",
        "when": [("P", "eq", "83350-45")],
        "gl": "65415-1800--CSI_U100-L529--",
    },
    {
        "id": "CSI-08",
        "when": [("P", "eq", "83350-46")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-09",
        "when": [("P", "eq", "83350-80")],
        "gl": "65415-5500--CSI_U100-L529--",
    },
    {
        "id": "CSI-10",
        "when": [("AM", "eq", "helpdesk911")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-11",
        "when": [("AM", "eq", "BBryanCSI1")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-12",
        "when": [("AM", "eq", "lc963")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-13",
        "when": [("AM", "eq", "mcard636")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-14",
        "when": [("AM", "eq", "MThomasCSI1")],
        "gl": "65420-3000--CSI_U100-L529--",
    },
    {
        "id": "CSI-15",
        "when": [("AM", "eq", "rquin19")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-16",
        "when": [("AM", "eq", "Sheana Elliott")],
        "gl": "65420-3000--CSI_U100-L528--",
    },
    {
        "id": "CSI-17",
        "when": [("AM", "eq", "TMutambayi1")],
        "gl": "65420-3000--CSI_U100-L528--",
    },
    {
        "id": "CSI-18",
        "when": [("AM", "eq", "vdh510")],
        "gl": "65415-2000--CSI_M100-L529--",
    },
    {
        "id": "CSI-19",
        "when": [("BP", "contains", "Cooper"), ("BX", "contains", "Cooper")],
        "gl": "Cooper to Cooper",
    },
    {
        "id": "CSI-20",
        "when": [("BW", "contains", "repair")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-21",
        "when": [("BW", "word", "RMA")],
        "gl": "65420-3000--CSI_U100-L529--",
    },
    {
        "id": "CSI-22",
        "when": [("BW", "contains", "Returns")],
        "gl": "65420-3000--CSI_U100-L529--",
    },
    {
        "id": "CSI-23",
        "when": [("BW", "contains", "Complaint")],
        "gl": "65420-3000--CSI_U100-L529--",
    },
    {
        "id": "CSI-24",
        "when": [
            ("BP", "contains", "Cooper"),
            ("BS", "contains", "Trumb"),
            ("BX", "contains", "Cooper"),
            ("CA", "contains", "Alajuela"),
        ],
        "gl": "51630-1000--CSI_M100-L510--",
    },
    {
        "id": "CSI-25",
        "when": [
            ("BP", "contains", "Cooper"),
            ("BS", "contains", "Worth"),
            ("BX", "contains", "Cooper"),
            ("CA", "contains", "Alajuela"),
        ],
        "gl": "51630-1000--CSI_M100-L510--",
    },
    {
        "id": "CSI-26",
        "when": [("CA", "contains", "Tonawanda"), ("BP", "not_contains", "Cooper")],
        "gl": "65440-8401--CSI_P100-L551--",
    },
    {
        "id": "CSI-27",
        "when": [("CA", "contains", "Sugar Land"), ("BP", "not_contains", "Cooper")],
        "gl": "65440-8401--CSI_M100-L560--",
    },
    {
        "id": "CSI-28",
        "when": [("C", "eq", "0000H4B189"), ("CD", "eq", "CA")],
        "gl": "65420-3000--CSI_U100-L527--",
    },
    {
        "id": "CSI-29",
        "when": [("C", "eq", "0000H93A28")],
        "gl": "65420-3000--CSI_U100-L527--",
    },
    {
        "id": "CSI-30",
        "when": [("BW", "contains", "help desk")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-31",
        "when": [("BW", "contains", "IT Dep")],
        "gl": "65415-1804--CSI_U100-L529--",
    },  # listed twice in the doc
    {
        "id": "CSI-32",
        "when": [("P|Q", "contains", "550-0"), ("CA", "contains", "Trumb")],
        "gl": "65440-8401--CSI_M100-L510--",
    },
    {
        "id": "CSI-33",
        "when": [("P|Q", "contains", "550-0"), ("CA", "contains", "Staff")],
        "gl": "65440-8401--CSI_M100-L562--",
    },
    {
        "id": "CSI-34",
        "when": [("P|Q", "contains", "550-0"), ("CA", "contains", "Worth")],
        "gl": "65440-8401--CSI_M100-L510--",
    },
    {
        "id": "CSI-35",
        "when": [("P", "contains", "Complaints")],
        "gl": "65415-1800--CSI_U100-L529--",
    },
    {
        "id": "CSI-36",
        "when": [("P", "contains", "Recall")],
        "gl": "65415-1800--CSI_U100-L529--",
    },
    {
        "id": "CSI-37",
        "when": [("P", "contains", "QA Dept")],
        "gl": "65415-8440--CSI_M100-L510--",
    },
    {
        "id": "CSI-38",
        "when": [
            ("P", "contains", "SO55000"),
            (SO_CITY, "contains", "Trumb"),
            ("BX", "not_contains", "Cooper"),
        ],
        "gl": "65420-3000--CSI_U100-L528--",
    },
    {
        "id": "CSI-39",
        "when": [
            ("P", "contains", "SO55000"),
            (SO_CITY, "contains", "Worth"),
            ("BX", "not_contains", "Cooper"),
        ],
        "gl": "65420-3000--CSI_U100-L527--",
    },
    {
        "id": "CSI-40",
        "when": [("BP", "contains", "Lynn Sales")],
        "gl": "65415-2113--CSI_U100-L529--",
    },
    {
        "id": "CSI-41",
        "when": [("BX", "contains", "Lynn Sales")],
        "gl": "65415-2113--CSI_U100-L529--",
    },
    {
        "id": "CSI-42",
        "when": [("CA", "contains", "Herten"), ("BP", "contains", "Cooper")],
        "gl": "51630-1000--CSI_M100-L510--",
    },
    # the last rule in the doc says "Column CA = SO55000", probably a typo for P, so it's the same as this one
    {
        "id": "CSI-43",
        "when": [
            ("P", "contains", "SO55000"),
            (SO_CITY, "contains", "Antonio"),
            ("BX", "not_contains", "Cooper"),
        ],
        "gl": "65420-3000--CSI_U100-L529--",
    },
]

RULES_V1["Belimo"] = [
    {"id": "BEL-01", "when": [("C", "contains", "9V1790")], "gl": "356200"},
    {"id": "BEL-02", "when": [("P", "contains", "sample")], "gl": "367211"},
    {"id": "BEL-03", "when": [("P", "contains", "Trade")], "gl": "367400"},
    {"id": "BEL-04", "when": [("P", "word", "Lit")], "gl": "367230"},
    {"id": "BEL-05", "when": [("P", "contains", "Promo")], "gl": "367230"},
    {"id": "BEL-06", "when": [("P", "word", "RMA")], "gl": "353590"},
    {"id": "BEL-07", "when": [("P", "contains", "2-10")], "gl": "367230"},
    {
        "id": "BEL-08",
        "when": [("BP", "contains", "Imprint"), ("BS", "contains", "Oshkosh")],
        "gl": "369600",
    },
    {
        "id": "BEL-09",
        "when": [("BY", "contains", "Imprint"), ("CA", "contains", "Oshkosh")],
        "gl": "369600",
    },  # BY is an address line, maybe meant BX
    {
        "id": "BEL-10",
        "when": [
            ("BP", "contains", "Belimo"),
            ("BS", "contains", "Danbury"),
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Sparks"),
        ],
        "gl": "356205",
    },
    {
        "id": "BEL-11",
        "when": [
            ("BP", "contains", "Belimo"),
            ("BS", "contains", "Sparks"),
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Danbury"),
        ],
        "gl": "356205",
    },
    {
        "id": "BEL-12",
        "when": [
            ("BP", "contains", "Warren"),
            ("BS", "contains", "Bethlehem"),
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Danbury"),
        ],
        "gl": "356205",
    },
    {
        "id": "BEL-13",
        "when": [
            ("BP", "contains", "Warren"),
            ("BS", "contains", "Bethlehem"),
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Sparks"),
        ],
        "gl": "356205",
    },
    {
        "id": "BEL-14",
        "when": [
            ("BP", "contains", "Belimo"),
            ("BS", "contains", "Danbury"),
            ("BX", "contains", "Warren"),
            ("CA", "contains", "Bethlehem"),
        ],
        "gl": "356205",
    },
    {
        "id": "BEL-15",
        "when": [
            ("BP", "contains", "Belimo"),
            ("BS", "contains", "Sparks"),
            ("BX", "contains", "Warren"),
            ("CA", "contains", "Bethlehem"),
        ],
        "gl": "356205",
    },
    {
        "id": "BEL-16",
        "when": [
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Danbury"),
            ("BP", "not_contains_any", ["Warren", "Belimo"]),
        ],
        "gl": "356200",
    },
    {
        "id": "BEL-17",
        "when": [
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Sparks"),
            ("BP", "not_contains_any", ["Warren", "Belimo"]),
        ],
        "gl": "356200",
    },
    {
        "id": "BEL-18",
        "when": [
            ("BX", "contains", "Warren"),
            ("CA", "contains", "Bethlehem"),
            ("BP", "not_contains_any", ["Warren", "Belimo"]),
        ],
        "gl": "356200",
    },
    {
        "id": "BEL-19",
        "enabled": False,
        "when": [
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Danbury"),
            ("BP", "not_contains_any", ["Warren", "Belimo"]),
        ],
        "gl": "353550",
    },
    {
        "id": "BEL-20",
        "enabled": False,
        "when": [
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Sparks"),
            ("BP", "not_contains_any", ["Warren", "Belimo"]),
        ],
        "gl": "353550",
    },
    {
        "id": "BEL-21",
        "enabled": False,
        "when": [
            ("BX", "contains", "Warren"),
            ("CA", "contains", "Bethlehem"),
            ("BP", "not_contains_any", ["Warren", "Belimo"]),
        ],
        "gl": "353550",
    },
    {"id": "BEL-22", "when": [("AJ", "contains", "ADC")], "gl": "353500"},
]

# if a row matches more than one rule, the one with more conditions wins (ties -> first in the doc)
# otherwise e.g. "Cooper to Cooper" would always beat the more specific Alajuela rule
for comp in RULES_V1:
    for r in RULES_V1[comp]:
        r["gl"] = normalize_gl(r["gl"])
        r["priority"] = len(r["when"])

In [6]:
pred_v1 = {}
for comp in raw:
    res = match_rules(raw[comp], RULES_V1[comp])
    existing = raw[comp][col_idx("M")].map(normalize_gl).values
    gl = np.where(
        existing != "", existing, res["gl"].values
    )  # keep values that were already in M
    src = np.where(
        existing != "",
        "existing",
        np.where(res["gl"].values != "", "rule", "unmatched"),
    )
    pred_v1[comp] = pd.DataFrame({"gl": gl, "source": src, "rule": res["rule"].values})
    print(comp)
    print(pd.Series(src).value_counts().to_string(), "\n")

Belimo
unmatched    7834
rule          871 

CSI
rule         2364
existing      727
unmatched     172 



## v1 vs answer key

In [7]:
for comp in pred_v1:
    p = pred_v1[comp]
    p["correct"] = p["gl"].values == truth[comp]
    print(comp, "overall accuracy: {:.1%}".format(p["correct"].mean()))
    print(p.groupby("source")["correct"].agg(["size", "mean"]).round(3), "\n")

Belimo overall accuracy: 7.5%
           size   mean
source                
rule        871  0.751
unmatched  7834  0.000 

CSI overall accuracy: 90.0%
           size   mean
source                
existing    727  1.000
rule       2364  0.935
unmatched   172  0.000 



In [8]:
# for each doc rule: of the rows it matches, how many agree with the answer key?
def rule_audit(comp, rules):
    rows = []
    for r in rules:
        if not r.get("enabled", True):
            continue
        m = rule_mask(raw[comp], r)
        if m.sum() == 0:
            continue
        t = pd.Series(truth[comp][m])
        rows.append(
            {
                "id": r["id"],
                "gl": r["gl"],
                "matched": int(m.sum()),
                "agree": round((t == r["gl"]).mean(), 2),
                "most common answer": t.value_counts().index[0],
            }
        )
    return pd.DataFrame(rows).sort_values("agree")


for comp in raw:
    print(comp)
    display(rule_audit(comp, RULES_V1[comp]))

Belimo


,id,gl,matched,agree,most common answer
6,BEL-07,367230,134,0.00,353590
11,BEL-14,356205,18,0.00,353500
4,BEL-05,367230,16,0.19,367210
2,BEL-03,367400,36,0.47,367400
0,BEL-01,356200,101,0.54,356200
13,BEL-17,356200,50,0.68,356200
12,BEL-16,356200,245,0.91,356200
1,BEL-02,367211,149,0.97,367211
14,BEL-22,353500,136,0.99,353500
3,BEL-04,367230,6,1.00,367230


CSI


,id,gl,matched,agree,most common answer
18,CSI-19,Cooper to Cooper,142,0.00,51630-1000--CSI_M100-L510--
4,CSI-05,65415-8401--CSI_M100-L510--,4,0.00,65410-3000--CSI_U100-L529--
11,CSI-12,65415-1804--CSI_U100-L529--,3,0.00,65415-5500--CSI_U100-L529--
12,CSI-13,65415-1804--CSI_U100-L529--,6,0.00,65415-5500--CSI_U100-L529--
32,CSI-34,65440-8401--CSI_M100-L510--,34,0.12,65420-3000--CSI_U100-L527--
20,CSI-21,65420-3000--CSI_U100-L529--,39,0.54,65420-3000--CSI_U100-L529--
31,CSI-33,65440-8401--CSI_M100-L562--,29,0.55,65440-8401--CSI_M100-L562--
24,CSI-26,65440-8401--CSI_P100-L551--,15,0.80,65440-8401--CSI_P100-L551--
39,CSI-42,51630-1000--CSI_M100-L510--,39,0.87,51630-1000--CSI_M100-L510--
33,CSI-35,65415-1800--CSI_U100-L529--,155,0.99,65415-1800--CSI_U100-L529--


## Looking for patterns

Main idea: add a "flow" column based on whether the sender (BP) and receiver (BX) are the company itself.

In [9]:
OWN = {"CSI": ["cooper"], "Belimo": ["belimo", "warren"]}

feat = {}
for comp in raw:
    df = raw[comp]
    f = pd.DataFrame(
        {
            c: get_col(df, c).values
            for c in [
                "C",
                "N",
                "P",
                "Q",
                "AJ",
                "AM",
                "AQ",
                "BP",
                "BS",
                "BW",
                "BX",
                "CA",
                "CD",
            ]
        }
    )
    pat = make_pattern(OWN[comp])
    sender_own = f["BP"].str.contains(pat, case=False, regex=True)
    receiver_own = f["BX"].str.contains(pat, case=False, regex=True)
    f["flow"] = np.select(
        [sender_own & receiver_own, sender_own, receiver_own],
        ["intercompany", "outbound", "inbound"],
        "third_party",
    )
    f["truth"] = truth[comp]
    feat[comp] = f

for comp in feat:
    print(comp)
    display(pd.crosstab(feat[comp]["truth"], feat[comp]["flow"], margins=True))

Belimo


flow,inbound,intercompany,outbound,third_party,All
truth,,,,,
353500,4,35,7731,136,7906
353550,0,0,7,0,7
353590,8,0,131,2,141
356200,258,0,0,0,258
356205,0,70,0,0,70
367210,0,0,13,0,13
367211,0,0,145,0,145
367230,0,0,6,0,6
367400,0,0,17,0,17


CSI


flow,inbound,intercompany,outbound,third_party,All
truth,,,,,
51630-1000--CSI_M100-L510--,6,48,6,0,60
65410-3000--CSI_U100-L529--,0,4,0,0,4
65415-1003--CSI_M100-L529--,0,0,14,0,14
65415-1800--CSI_U100-L529--,182,0,147,6,335
65415-1804--CSI_U100-L529--,44,27,37,0,108
65415-2000--CSI_M100-L529--,0,0,34,0,34
65415-2113--CSI_U100-L529--,16,0,0,11,27
65415-5500--CSI_U100-L529--,0,39,3,2,44
65415-6000--CSI_U100-L529--,0,0,9,0,9


In [10]:
def gl_profile(
    comp,
    gl,
    cols=["C", "AJ", "AQ", "AM", "BP", "BS", "BW", "BX", "CA", "CD", "P"],
    top=5,
):
    # most common values of each column for one GL code
    d = feat[comp][feat[comp]["truth"] == gl]
    out = {}
    for c in cols:
        vc = d[c].replace("", "<blank>").value_counts().head(top)
        out[c] = ", ".join(f"{k} ({v})" for k, v in vc.items())
    return pd.Series(out, name=f"{gl} ({len(d)} rows)").to_frame()


display(gl_profile("Belimo", "356205"))
display(gl_profile("Belimo", "369600"))
display(gl_profile("CSI", "51630-1000--CSI_M100-L510--"))

,356205 (70 rows)
C,"97644 (52), 00009V1790 (14), 000088W..."
AJ,"IMP (31), WWS (17), ISS (12), TP (6)..."
AQ,"ZZZ (49), FRT (21)"
AM,"<blank> (58), belimoserp (12)"
BP,"BELIMO US, Inc. (46), BELIMO SENSORS..."
BS,"Danbury (46), DORVAL (14), BETHLEHEM..."
BW,"RECEIVING (19), BELIMO CUSTOMIZATION..."
BX,"BELIMO US, Inc. (24), BELIMO AUTOMAT..."
CA,"Danbury (24), MUMBAI (12), SPARKS (1..."
CD,"US (34), CA (13), IN (12), CH (8), L..."


,369600 (142 rows)
C,"97644 (138), 543548689 (4)"
AJ,"MAN (60), TP (54), SCC (19), ISS (4)..."
AQ,"ZZZ (96), FRT (46)"
AM,"<blank> (138), belimoserp (4)"
BP,"BELIMO US, Inc. (73), 4imprint (51),..."
BS,"Danbury (76), Oshkosh (51), OSHKOSH ..."
BW,"Marissa Palmer (60), Rebecca Jacobs..."
BX,"BELIMO US, Inc. (74), Belimo Air Con..."
CA,"Danbury (74), SPARKS (16), BOULDER (..."
CD,US (142)


,51630-1000--CSI_M100-L510-- (60 rows)
C,"0000H4B189 (34), 38320 (24), 0000208..."
AJ,"WWS (37), SCC (15), TP (6), ADC (2)"
AQ,"ZZZ (48), FRT (12)"
AM,<blank> (60)
BP,"CooperSurgical, Inc. (28), CooperSur..."
BS,"Trumbull (28), Ft. Worth (19), CELIN..."
BW,"BILLING (21), <blank> (10), MARGARIT..."
BX,"COOPERSURGICAL DISTRIB B.V. (29), CO..."
CA,"HERTEN (34), ALAJUELA (14), HOUSTON ..."
CD,"NL (34), CR (14), US (12)"


In [11]:
# CSI inbound: receiving city decides the location part of the code
f = feat["CSI"]
inb = f[f["flow"] == "inbound"]
ct = pd.crosstab(inb["CA"].str.upper(), inb["truth"])
ct[ct.sum(axis=1) >= 4]

truth,51630-1000--CSI_M100-L510--,65415-1800--CSI_U100-L529--,65415-1804--CSI_U100-L529--,65415-2113--CSI_U100-L529--,65420-3000--CSI_U100-L527--,65420-3000--CSI_U100-L529--,65440-8401--CSI_M100-L510--,65440-8401--CSI_M100-L560--,65440-8401--CSI_M100-L562--,65440-8401--CSI_P100-L551--,Split-65440-8401--CSI_M100-L510--
CA,,,,,,,,,,,
ALAJUELA,6,0,0,0,0,0,0,0,0,0,0
CLANTON,0,0,0,8,0,0,0,0,0,0,0
FT. WORTH,0,35,0,0,2,13,4,0,0,0,0
HOUSTON,0,0,0,4,0,0,0,0,0,0,0
NORTH TONAWANDA,0,3,0,0,0,0,0,0,0,12,0
STAFFORD,0,0,0,0,0,0,13,0,8,0,0
SUGAR LAND,0,0,0,0,0,0,0,30,0,0,0
TRUMBULL,0,144,44,0,0,11,223,0,0,0,22
WHITE LAKE,0,0,0,4,0,0,0,0,0,0,0


In [12]:
# do adjustment lines (SCC, CHBK ...) have the same GL as the original shipment (same tracking #)?
for comp in feat:
    f = feat[comp]
    adj = f["AJ"].isin(["SCC", "CHBK", "RADJ", "ADC", "GCC"])
    orig = (
        f[~adj & (f["N"] != "")]
        .groupby("N")["truth"]
        .agg(lambda s: s.iloc[0] if s.nunique() == 1 else None)
        .dropna()
    )
    m = f.loc[adj, "N"].map(orig)
    found = m.notna()
    same = (m[found] == f.loc[adj, "truth"][found]).mean()
    print(
        f"{comp}: {adj.sum()} adjustment rows, {found.sum()} have the original in this file, same GL: {same:.1%}"
    )

Belimo: 508 adjustment rows, 436 have the original in this file, same GL: 99.1%
CSI: 1530 adjustment rows, 729 have the original in this file, same GL: 97.0%


### What I found

Both companies follow the same structure. Each layer below overrides the one above it:

0. **Default.** Outbound to customers. CSI = `65420-3000--CSI_U100-L527--` (DFW), Belimo = `353500`.
1. **Direction + site.**
   - Supplier to us: CSI `65440-8401` with the location from the receiving site (L510 / L560 / L562 / P100-L551); Belimo `356200`.
   - Between our own sites or affiliates: CSI `51630-1000` (Herten / Alajuela); Belimo `356205`.
   - Belimo export to a Canadian customer: `353550`. This is what the "and Column CD" rules meant (CD = CA).
2. **Charge type.** UPS service / audit / print charges = `65415-5500`; imports = `Split-65440-8401...`.
3. **Purpose.** Reference (P), attention (BW), UPS user (AM): complaints, IT, RMA, samples, trade shows, marketing.
4. **Adjustment lines** (SCC etc.) follow the original shipment with the same tracking #.

Places where the answer key doesn't match the doc:
- CSI: there is no "Cooper to Cooper" in the answers.
- CSI: P = 83350-15 is `65410-3000--CSI_U100-L529--`; user lc963 / mcard636 are `65415-5500`.
- CSI: Stafford purchases are L510 when the PO550 number is in P, and L562 when it's only in Q.
- CSI: BW "RMA ... RECALL" is a recall (`65415-1800`), not a return.
- Belimo: 2-10xxxx is an RMA number, so it's `353590`, not `367230`.
- Belimo trade shows: lit = `367230`, promo = `367210`, otherwise `367400`.
- Belimo: Belimo -> Warren is `353500`. Only Warren -> Belimo is `356205`.

Client answers (10/15/26), now in the v2 rules. Where the client picked the doc, accuracy against this answer key goes down on purpose:
- CSI Q1/Q6: Cooper to Cooper (incl. Herten with a sales order) = `INTERCOMPANY`, their team picks the code (`C-COOP`).
- CSI Q2: PO550 + receiving city Fort Worth = `65440-8401--CSI_M100-L510--`, doc rule (`C-PO-WORTH`).
- CSI Q3: anything to or from Stafford = L562, PO doesn't matter (`C-STF`); from a Cooper facility = `INTERCOMPANY` (`C-COOP-STF`).
- CSI Q4: RMA attention line + complaints reference, the reference wins (`C-RMA-COMPL`).
- Belimo Q10: Warren is a Belimo site, so Belimo -> Warren = `356205` (`B-ICO`).
- Belimo Q11: promo = `367210`, lit / literature = `367230`, other trade show = `367400`.
- Belimo Q12: account 9V1790 = `356200`, doc rule (`B-ACCT`).
- Still open (talk on the call): Q7 users lc963 / mcard636, Q15 Split and `65415-1003`.

## v2 rules

In [13]:
OWN = {
    "CSI": ["cooper"],
    "Belimo": ["belimo", "warren"],
}  # words that mean "our company" in BP/BX

U527 = "65420-3000--CSI_U100-L527--"
U528 = "65420-3000--CSI_U100-L528--"
U529R = "65420-3000--CSI_U100-L529--"
COMPL = "65415-1800--CSI_U100-L529--"
IT = "65415-1804--CSI_U100-L529--"
SVC = "65415-5500--CSI_U100-L529--"
PUR = "65440-8401--CSI_M100-L510--"
ICO = "51630-1000--CSI_M100-L510--"
INTERCO = "INTERCOMPANY"  # client vets these by hand (Q1, 10/15/26)
FROM_SUPPLIER = [
    ("BX", "contains_any", OWN["CSI"]),
    ("BP", "not_contains_any", OWN["CSI"]),
    ("AJ", "in", ["FC", "TP"]),
]

# layer: 0 default, 1-2 direction + site, 3 charge type, 4+ purpose (dept, person, reference)
# higher layer wins
RULES_V2 = {}
RULES_V2["CSI"] = [
    {
        "id": "C-DEF",
        "layer": 0,
        "when": [],
        "gl": U527,
        "why": "default: DFW outbound to customers",
    },
    {
        "id": "C-OUT-TRB",
        "layer": 1,
        "when": [("BP", "contains_any", OWN["CSI"]), ("BS", "contains", "Trumb")],
        "gl": U528,
        "why": "shipped from Trumbull",
    },
    {
        "id": "C-OUT-TON",
        "layer": 1,
        "when": [("BP", "contains_any", OWN["CSI"]), ("BS", "contains", "Tonawanda")],
        "gl": COMPL,
        "why": "shipped from North Tonawanda (complaints/recalls)",
    },
    {
        "id": "C-IN-RET",
        "layer": 1,
        "when": [
            ("BX", "contains_any", OWN["CSI"]),
            ("BP", "not_contains_any", OWN["CSI"]),
        ],
        "gl": COMPL,
        "why": "sent to Cooper by outside party, assume customer return",
    },
    {
        "id": "C-IN-PUR-TRB",
        "layer": 2,
        "when": FROM_SUPPLIER + [("CA", "contains_any", ["Trumb", "Worth"])],
        "gl": PUR,
        "why": "supplier to Trumbull/DFW",
    },
    {
        "id": "C-IN-PUR-SGL",
        "layer": 2,
        "when": FROM_SUPPLIER + [("CA", "contains", "Sugar Land")],
        "gl": "65440-8401--CSI_M100-L560--",
        "why": "supplier to Sugar Land",
    },
    {
        "id": "C-STF",
        "layer": 2,
        "when": [("BS|CA", "contains", "Stafford")],
        "gl": "65440-8401--CSI_M100-L562--",
        "why": "to or from Stafford, PO doesn't matter (client Q3)",
    },
    {
        "id": "C-IN-PUR-TON",
        "layer": 2,
        "when": FROM_SUPPLIER + [("CA", "contains", "Tonawanda")],
        "gl": "65440-8401--CSI_P100-L551--",
        "why": "supplier to North Tonawanda",
    },
    {
        "id": "C-IMP",
        "layer": 2,
        "when": [("BX", "contains_any", OWN["CSI"]), ("AJ", "eq", "IMP")],
        "gl": "Split-" + PUR,
        "why": "import -> split",
    },
    {
        "id": "C-ICO-INTL",
        "layer": 2,
        "when": [("BX", "contains_any", OWN["CSI"]), ("CD", "in", ["NL", "CR"])],
        "gl": ICO,
        "why": "to Cooper entity abroad (Herten / Alajuela)",
    },
    {
        "id": "C-SVC",
        "layer": 3,
        "when": [("AQ", "in", ["SVC", "AUD", "PRT"])],
        "gl": SVC,
        "why": "UPS service / audit / print charge",
    },
    {
        "id": "C-TAX",
        "layer": 3,
        "when": [("AQ", "eq", "TAX")],
        "gl": U527,
        "why": "tax",
    },
    {
        "id": "C-SO-TRB",
        "layer": 4,
        "when": [("P", "contains", "SO55000"), ("BS", "contains", "Trumb")],
        "gl": U528,
        "why": "Trumbull sales order",
    },
    {
        "id": "C-SO-DFW",
        "layer": 4,
        "when": [("P", "contains", "SO55000"), ("BS", "contains", "Worth")],
        "gl": U527,
        "why": "DFW sales order (incl. overseas distributor)",
    },
    {
        "id": "C-PO-WORTH",
        "layer": 5,
        "when": [("P|Q", "contains", "550-0"), ("CA", "contains", "Worth")],
        "gl": PUR,
        "why": "PO550 number, receiving city Fort Worth (doc rule, client Q2)",
    },
    {
        "id": "C-IT-ACCT",
        "layer": 4,
        "when": [("C", "eq", "0000XH8253")],
        "gl": IT,
        "why": "IT UPS account",
    },
    {
        "id": "C-IT-BW",
        "layer": 4,
        "when": [("BW", "contains_any", ["IT Dep", "help desk", "repair"])],
        "gl": IT,
        "why": "IT dept / repair",
    },
    {
        "id": "C-IT-AM",
        "layer": 4,
        "when": [("AM", "in", ["helpdesk911", "BBryanCSI1", "rquin19", "EdMak"])],
        "gl": IT,
        "why": "IT user id",
    },
    {
        "id": "C-P-1804",
        "layer": 4,
        "when": [("P", "eq", "83350-46")],
        "gl": IT,
        "why": "reference 83350-46",
    },
    {
        "id": "C-SVC-AM",
        "layer": 4,
        "when": [("AM", "in", ["mcard636", "lc963", "A.MollicaCSI"])],
        "gl": SVC,
        "why": "user id (answer key says 5500, doc said 1804)",
    },
    {
        "id": "C-P-5500",
        "layer": 4,
        "when": [("P", "eq", "83350-80")],
        "gl": SVC,
        "why": "reference 83350-80",
    },
    {
        "id": "C-RMA",
        "layer": 4,
        "when": [("BW", "word", ["RMA", "Returns", "Complaint"])],
        "gl": U529R,
        "why": "RMA / returns dept",
    },
    {
        "id": "C-RMA-COMPL",
        "layer": 4,
        "when": [
            ("BW", "word", ["RMA", "Returns", "Complaint"]),
            ("P", "contains_any", ["Complaint", "Recall", "83350-45"]),
        ],
        "gl": COMPL,
        "why": "RMA attention line but complaints reference, reference wins (client Q4)",
    },
    {
        "id": "C-RMA-RECALL",
        "layer": 5,
        "when": [("BW", "contains", "RECA")],
        "gl": COMPL,
        "why": "RMA ... RECALL = recall",
    },
    {
        "id": "C-RMA-AM",
        "layer": 4,
        "when": [("AM", "in", ["MThomasCSI1"])],
        "gl": U529R,
        "why": "returns user id",
    },
    {
        "id": "C-L528-AM",
        "layer": 4,
        "when": [("AM", "in", ["Sheana Elliott", "TMutambayi1"])],
        "gl": U528,
        "why": "user id",
    },
    {
        "id": "C-COMPL",
        "layer": 4,
        "when": [("P", "contains_any", ["Complaint", "Recall", "83350-45"])],
        "gl": COMPL,
        "why": "complaint / recall",
    },
    {
        "id": "C-QA",
        "layer": 5,
        "when": [("P", "contains", "QA Dept")],
        "gl": "65415-8440--CSI_M100-L510--",
        "why": "QA dept",
    },
    {
        "id": "C-P-6000",
        "layer": 4,
        "when": [("P", "eq", "83350-60")],
        "gl": "65415-6000--CSI_U100-L529--",
        "why": "reference 83350-60",
    },
    {
        "id": "C-P-15",
        "layer": 4,
        "when": [("P", "eq", "83350-15")],
        "gl": "65410-3000--CSI_U100-L529--",
        "why": "answer key differs from doc (doc: 65415-8401--CSI_M100-L510--)",
    },
    {
        "id": "C-SHOW",
        "layer": 4,
        "when": [
            (
                "AM|BW|BX",
                "contains_any",
                ["vdh510", "Tara Luczak", "Maria Tobin", "Drake Hotel", "AUGS"],
            )
        ],
        "gl": "65415-2000--CSI_M100-L529--",
        "why": "trade show / conference",
    },
    {
        "id": "C-LYNN",
        "layer": 4,
        "when": [("BP|BX", "contains", "Lynn Sales")],
        "gl": "65415-2113--CSI_U100-L529--",
        "why": "AL-Lynn Sales rep",
    },
    {
        "id": "C-COOP",
        "layer": 9,
        "when": [
            ("BP", "contains_any", OWN["CSI"]),
            ("BX", "contains_any", OWN["CSI"]),
            ("BS", "ne_col", "CA"),  # same city = UPS account charge, not a shipment
        ],
        "gl": INTERCO,
        "why": "Cooper to Cooper, client team picks the intercompany code (client Q1/Q6)",
    },
    {
        "id": "C-COOP-STF",
        "layer": 9,
        "when": [
            ("BP", "contains_any", OWN["CSI"]),
            ("CA", "contains", "Stafford"),
            ("BS", "ne_col", "CA"),
        ],
        "gl": INTERCO,
        "why": "Cooper facility to Stafford (client Q3)",
    },
]

RULES_V2["Belimo"] = [
    {
        "id": "B-DEF",
        "layer": 0,
        "when": [],
        "gl": "353500",
        "why": "default: outbound to customers",
    },
    {
        "id": "B-OUT-CA",
        "layer": 1,
        "when": [("CD", "eq", "CA"), ("BX", "not_contains_any", OWN["Belimo"])],
        "gl": "353550",
        "why": "export to Canada (not an affiliate)",
    },
    {
        "id": "B-IN",
        "layer": 1,
        "when": [
            ("BX", "contains_any", OWN["Belimo"]),
            ("BP", "not_contains_any", OWN["Belimo"]),
        ],
        "gl": "356200",
        "why": "supplier to Belimo",
    },
    {
        "id": "B-ICO",
        "layer": 2,
        "when": [
            ("BX", "contains_any", OWN["Belimo"]),
            ("BP", "contains_any", OWN["Belimo"]),
            ("BS", "ne_col", "CA"),
        ],
        "gl": "356205",
        "why": "between Belimo sites / affiliates, Warren counts as a Belimo site (client Q10)",
    },
    {
        "id": "B-ACCT",
        "layer": 3,
        "when": [("C", "contains", "9V1790")],
        "gl": "356200",
        "why": "UPS account 9V1790 is inbound (doc rule, client Q12)",
    },
    {
        "id": "B-ACCT-SAMPLE",
        "layer": 4,
        "when": [("C", "contains", "9V1790"), ("P", "contains", "sample")],
        "gl": "356200",
        "why": "account 9V1790 wins over samples; marketing / RMA still win (client Q12/Q13)",
    },
    {
        "id": "B-MKT",
        "layer": 3,
        "when": [("BP", "contains", "imprint")],
        "gl": "369600",
        "why": "marketing: 4imprint",
    },
    {
        "id": "B-MKT2",
        "layer": 3,
        "when": [
            ("BW|P", "contains_any", ["Marissa", "BUSINES CARD", "Field Support"])
        ],
        "gl": "369600",
        "why": "marketing people / business cards",
    },
    {
        "id": "B-RMA",
        "layer": 4,
        "when": [("P", "word", "RMA")],
        "gl": "353590",
        "why": "RMA",
    },
    {
        "id": "B-RMA2",
        "layer": 4,
        "when": [("P", "contains", "2-10")],
        "gl": "353590",
        "why": "2-10xxxx is an RMA number (doc said 367230)",
    },
    {
        "id": "B-SAMPLE",
        "layer": 4,
        "when": [("P", "contains", "sample")],
        "gl": "367211",
        "why": "samples",
    },
    {
        "id": "B-TRADE",
        "layer": 4,
        "when": [("P", "contains", "trade")],
        "gl": "367400",
        "why": "trade show",
    },
    {
        "id": "B-PROMO",
        "layer": 5,
        "when": [("P", "contains", "promo")],
        "gl": "367210",
        "why": "promo / promotions (client Q11)",
    },
    {
        "id": "B-LIT",
        "layer": 6,
        "when": [("P", "word", ["lit", "literature"])],
        "gl": "367230",
        "why": "lit / literature (client Q11)",
    },
]

# priority = layer * 10 + number of conditions
for comp in RULES_V2:
    for r in RULES_V2[comp]:
        r["gl"] = normalize_gl(r["gl"])
        r["priority"] = r["layer"] * 10 + len(r["when"])
        for cols, op, val in r["when"]:
            assert op in OPS, r["id"]
print({k: len(v) for k, v in RULES_V2.items()})

{'CSI': 34, 'Belimo': 14}


In [14]:
# adjustment-type lines (shipping charge corrections etc.) follow the original shipment
ADJUSTMENT_TYPES = ["SCC", "CHBK", "RADJ", "ADC", "GCC"]
PROTECTED_RULES = ["C-SVC", "C-TAX"]  # these keep their own GL


def inherit_from_tracking(raw, gl, rule, ledger=None):
    gl = gl.copy()
    rule = rule.copy()
    trk = get_col(raw, "N").values
    adj = np.isin(get_col(raw, "AJ").values, ADJUSTMENT_TYPES)

    base = pd.DataFrame({"N": trk, "GL": gl, "R": rule})[~adj & (trk != "")]
    src_rule = base.drop_duplicates("N").set_index("N")["R"]
    if ledger is not None and len(ledger) > 0:
        base = pd.concat([base, ledger[["N", "GL"]]])
    n_gl = base.groupby("N")["GL"].nunique()
    one = (
        base[base["N"].isin(n_gl[n_gl == 1].index)]
        .drop_duplicates("N")
        .set_index("N")["GL"]
    )

    new_gl = pd.Series(trk).map(one).values
    ok = adj & pd.notna(new_gl) & ~np.isin(rule, PROTECTED_RULES)
    gl[ok] = new_gl[ok]
    rule[ok] = "TRACKING"
    from_rule = np.where(ok, pd.Series(trk).map(src_rule).fillna("history").values, "")
    multi = adj & pd.Series(trk).isin(n_gl[n_gl > 1].index).values
    return gl, rule, from_rule, multi

In [15]:
pred_v2 = {}
for comp in raw:
    res = match_rules(raw[comp], RULES_V2[comp])
    gl, rule, from_rule, multi = inherit_from_tracking(
        raw[comp], res["gl"].values.astype(object), res["rule"].values.astype(object)
    )
    pred_v2[comp] = pd.DataFrame({"gl": gl, "rule": rule, "inherited_from": from_rule})

summary = []
for comp in raw:
    summary.append(
        {
            "company": comp,
            "v1 accuracy": round((pred_v1[comp]["gl"].values == truth[comp]).mean(), 4),
            "v2 accuracy": round((pred_v2[comp]["gl"].values == truth[comp]).mean(), 4),
            "v2 wrong rows": int((pred_v2[comp]["gl"].values != truth[comp]).sum()),
        }
    )
pd.DataFrame(summary)

,company,v1 accuracy,v2 accuracy,v2 wrong rows
0,Belimo,0.0751,0.9912,77
1,CSI,0.9001,0.9473,172


In [16]:
# accuracy per rule and the rows that are still wrong
for comp in raw:
    p = pred_v2[comp].copy()
    p["correct"] = p["gl"].values == truth[comp]
    print(comp)
    display(
        p.groupby("rule")["correct"].agg(["size", "mean"]).sort_values("mean").head(10)
    )

    wrong = feat[comp][~p["correct"].values].assign(
        pred=p["gl"][~p["correct"]].values, rule=p["rule"][~p["correct"]].values
    )
    display(
        wrong.groupby(["rule", "pred", "truth", "AJ", "BP", "BX"])
        .size()
        .rename("rows")
        .sort_values(ascending=False)
        .head(15)
        .reset_index()
    )
    wrong.to_csv(OUTPUT_DIR / f"{comp}_v2_wrong_rows.csv", index=False)

Belimo


,size,mean
rule,,
B-ACCT,97,0.525773
B-ICO,70,0.742857
B-IN,201,0.980100
TRACKING,436,0.990826
B-DEF,7464,0.999330
B-ACCT-SAMPLE,2,1.000000
B-LIT,6,1.000000
B-MKT,51,1.000000
B-MKT2,67,1.000000


,rule,pred,truth,AJ,BP,BX,rows
0,B-ACCT,356200,353500,TP,"BELIMO US, Inc.",AKRON ELECTRIC,18
1,B-ICO,356205,353500,ISS,"BELIMO US, Inc.",WARREN CONTROLS CORP.,18
2,B-ACCT,356200,356205,IMP,BELIMO SENSORS INC.,"BELIMO US, Inc.",14
3,B-ACCT,356200,353500,TP,"BELIMO US, Inc.",AB ELECTRONICS,12
4,B-DEF,353500,369600,,"BELIMO US, Inc.",JW MARRIOTT BASE NASHVILLE,4
5,B-IN,356200,353500,RTS,,"BELIMO US, Inc.",4
6,TRACKING,356200,369600,SCC,"BELIMO US, Inc.","BELIMO US, Inc.",4
7,B-ACCT,356200,353500,SVCH,"BELIMO US, Inc.","BELIMO US, Inc.",2
8,B-DEF,353500,369600,CADJ,"BELIMO US, Inc.","BELIMO US, Inc.",1


CSI


,size,mean
rule,,
C-COOP,59,0.000000
C-COOP-STF,2,0.000000
C-PO-WORTH,32,0.062500
C-IN-RET,6,0.333333
C-OUT-TRB,19,0.368421
C-STF,27,0.518519
TRACKING,729,0.934156
C-DEF,581,0.994836
C-COMPL,285,0.996491


,rule,pred,truth,AJ,BP,BX,rows
0,C-COOP,INTERCOMPANY,51630-1000--CSI_M100-L510--,WWS,"CooperSurgical, Inc. DFW",COOPERSURGICAL DISTRIB B.V.,16
1,C-PO-WORTH,65440-8401--CSI_M100-L510--,65420-3000--CSI_U100-L527--,MAN,"CooperSurgical, Inc. DFW",TX HLTH FORT WORTH HOSPITAL,14
2,C-STF,65440-8401--CSI_M100-L562--,65440-8401--CSI_M100-L510--,FC,AstroNova Inc,COOPER SURGICAL,10
3,C-PO-WORTH,65440-8401--CSI_M100-L510--,65420-3000--CSI_U100-L527--,MAN,"CooperSurgical, Inc. DFW",TX HLTH SW FORT WORTH HSP,8
4,C-COOP,INTERCOMPANY,51630-1000--CSI_M100-L510--,WWS,"CooperSurgical, Inc.",COOPERSURGICAL DISTRIB B.V.,8
5,C-COOP,INTERCOMPANY,51630-1000--CSI_M100-L510--,WWS,"CooperSurgical, Inc.","COOPER MEDICAL, SRL",6
6,TRACKING,65440-8401--CSI_M100-L510--,65420-3000--CSI_U100-L527--,SCC,"CooperSurgical, Inc. DFW",TX HLTH SW FORT WORTH HSP,6
7,C-COOP,INTERCOMPANY,65415-5500--CSI_U100-L529--,ISS,"CooperSurgical, Inc.",Cooper Surgical,5
8,C-COOP,INTERCOMPANY,65420-3000--CSI_U100-L527--,WWS,"CooperSurgical, Inc. DFW",COOPERSURGICAL DISTRIB B.V.,5
9,C-COOP,INTERCOMPANY,65415-1804--CSI_U100-L529--,ISS,"CooperSurgical, Inc.",CooperGenomics,5


Notes:
- v2 was written while looking at this answer key, so the accuracy is in-sample and probably too high. The real test is the next batch.
- Most of the rows that are still wrong look the same as other rows but have a different answer, e.g. CSI IT shipments where the SCC correction line got 65415-1003. I need to ask accounting about these.
- A few rules use people's names (Marissa Palmer, Tara Luczak) and will need updating if people change.